[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/26_lora.ipynb)

# 🟠 Medium: LoRA (Low-Rank Adaptation)

Implement **LoRA** — parameter-efficient fine-tuning for large models.

$$h = W_0 x + \frac{\alpha}{r} B A x$$

### Signature
```python
class LoRALinear(nn.Module):
    def __init__(self, in_features, out_features, rank, alpha=1.0): ...
    def forward(self, x: Tensor) -> Tensor: ...
```

### Requirements
- `self.linear`: frozen `nn.Linear` (weight & bias `requires_grad=False`)
- `self.lora_A`: `nn.Parameter(rank, in_features)` — random init
- `self.lora_B`: `nn.Parameter(out_features, rank)` — **zero** init
- Scaling: `alpha / rank`

In [46]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


In [47]:
import torch
import torch.nn as nn

In [48]:
# ✏️ YOUR IMPLEMENTATION HERE

class LoRALinear(nn.Module):
    def __init__(self, in_features, out_features, rank, alpha=1.0):
        super().__init__()
        # pass  # frozen linear + lora_A + lora_B
        self.in_features = in_features
        self.out_features = out_features
        self.r = rank
        self.a = alpha
        # always (in_features, out_features) for nn.Linear
        # since it's a wrapper
        self.linear = nn.Linear(in_features, out_features)
        # set the sets of (weight, bias) to frozen
        self.linear.weight.requires_grad = False
        self.linear.bias.requires_grad = False

        # for nn.Parameter, it's a tensor, so no need to follow
        # any convention
        self.lora_A = nn.Parameter(torch.randn(rank, in_features), requires_grad = True)
        self.lora_B = nn.Parameter(torch.zeros(out_features, rank), requires_grad = True)


    def forward(self, x):
        # pass  # base + lora
        base = self.linear(x)
        lora = (self.a / self.r) *  x @ self.lora_A.T @ self.lora_B.T
        return base + lora
        # return


In [49]:
# 🧪 Debug
layer = LoRALinear(16, 8, rank=4)
x = torch.randn(2, 16)
print('Output:', layer(x).shape)
print('Trainable:', sum(p.numel() for p in layer.parameters() if p.requires_grad))
print('Total:    ', sum(p.numel() for p in layer.parameters()))

Output: torch.Size([2, 8])
Trainable: 96
Total:     232


In [50]:
# ✅ SUBMIT
from torch_judge import check
check('lora')


🧪 Testing: LoRA (Low-Rank Adaptation) (Medium)
──────────────────────────────────────────────────
  ✅ [1/5] Base weights frozen (1.1ms)
  ✅ [2/5] LoRA parameter shapes (0.7ms)
  ✅ [3/5] B=0 means output equals base (3.5ms)
  ✅ [4/5] Only LoRA params get gradients (1.3ms)
  ✅ [5/5] Forward computation (1.7ms)
──────────────────────────────────────────────────
  🎉 All 5 tests passed! (8.3ms total)
  Progress saved. Run status() to see your dashboard.

